In [1]:
import pandas as pd
import re 
import emoji 
import html
from tqdm import tqdm
import nltk
from nltk.corpus import stopwords

nltk.download('stopwords')

tqdm.pandas()
print("libraries success loaded")

libraries success loaded


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\ASUS\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [2]:
df = pd.read_csv("rawData/dataX.csv")
df = df[['created_at', 'user_id_str', 'full_text']]
df['full_text'] = df['full_text'].fillna('')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3714 entries, 0 to 3713
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   created_at   3714 non-null   object
 1   user_id_str  3714 non-null   int64 
 2   full_text    3714 non-null   object
dtypes: int64(1), object(2)
memory usage: 87.2+ KB


In [3]:
initial_count = len(df)
df.drop_duplicates(subset=['full_text'], inplace=True)
df.reset_index(drop=True, inplace=True)

print(f"Data loaded: {len(df)} tweets (removed {initial_count - len(df)} duplicates)")
print(f"Columns: {df.columns.tolist()}")

Data loaded: 1882 tweets (removed 1832 duplicates)
Columns: ['created_at', 'user_id_str', 'full_text']


In [4]:
def clean_tweet_bertopic(text):
    if not isinstance(text, str):
        return ""
    
    # decode html
    text = html.unescape(text)

    # convert emoji to text
    text = emoji.demojize(text, language='en')
    text = re.sub(r':(\w+):', r'\1', text)

    # remove url
    text = re.sub(r'http\S+|https\S+', '', text)

    # remove mention
    text = re.sub(r'@|w+', '', text)

    # remove symbol
    text = re.sub(r'#(\w)', r'\1', text)

    # remove retweet indicator
    text = re.sub(r'^RT[\s]+', '', text)

    # remove spesific twitter artifact
    text = re.sub(r'&\w+;', '', text)

    # keep numbers but remove currency
    text = re.sub(r'[^\w\s\.,?!]', ' ', text)

    # clean whitespace
    text = re.sub(r'\s+', ' ', text).strip()

    # case folding
    text = text.lower()

    return text

df['clean_text'] = df['full_text'].progress_apply(clean_tweet_bertopic)
print('Cleaning and case folding completed')

100%|██████████| 1882/1882 [00:00<00:00, 7710.71it/s]

Cleaning and case folding completed


In [5]:
def normalize_minimal(text):
    critical_norm = {
        'yg': 'yang',
        'dg': 'dengan',
        'dgn': 'dengan',
        'ga': 'tidak',
        'gak': 'tidak',
        'tdk': 'tidak',
        'krn': 'karena',
        'sdh': 'sudah',
        'udh': 'sudah',
        'bs': 'bisa',
        'tp': 'tapi',
        'utk': 'untuk',
        'banget': 'sangat',
        'bgt': 'sangat',
        'deh': '', 
        'sih': '',
        'dong': '',
        'ya': '',
        'kok': '',
        'gw': 'saya',
        'gue': 'saya',
        'lu': 'kamu',
        'loe': 'kamu',
        'elo': 'kamu',
    }

    words = text.split()
    normalized_words = []

    for word in words:
        if word in critical_norm:
            replacement = critical_norm[word]
            if replacement:
                normalized_words.append(replacement)
        else:
            normalized_words.append(word)
    return ' '.join(normalized_words)

df['normalized'] = df['clean_text'].progress_apply(normalize_minimal)
print("minimal normalization completed")
print(f"normalization dictionary size: 20 critical terms")

100%|██████████| 1882/1882 [00:00<00:00, 179031.55it/s]

minimal normalization completed
normalization dictionary size: 20 critical terms


In [6]:
def remove_minimal_stopwords(text):
    minimal_stopwords = {
        'yang', 'dan', 'di', 'ke', 'dari', 'untuk', 'pada', 'dengan',
        'oleh', 'karena', 'atau', 'telah', 'akan', 'dalam', 'ini', 'itu',
        'juga', 'ada', 'adalah', 'merupakan', 'tersebut', 'sebuah',
        'saya', 'kamu', 'dia', 'mereka', 'kita', 'kami', 'kalian',
        'ini', 'itu', 'sini', 'situ', 'sana',
        'tapi', 'tetapi', 'namun', 'lagi', 'pun', 'lah', 'kah',
        'ya', 'tidak', 'bukan', 'jangan', 'sudah', 'belum',
        'sangat', 'sekali', 'agak', 'cukup', 'hanya', 'hanyalah',
        'adalah', 'ialah', 'merupakan',
        'jika', 'kalau', 'apabila', 'kapan', 'dimana', 'kemana',
        'berapa', 'bagaimana', 'mengapa', 'kenapa'
    }

    words = text.split()
    filtered_words = [word for word in words if word not in minimal_stopwords]
    return ' '.join(filtered_words)

df['final_bertopic'] = df['normalized'].progress_apply(remove_minimal_stopwords)

min_length = 15
initial_len = len(df)
df = df[df['final_bertopic'].str.len() >= min_length]
df.reset_index(drop=True, inplace=True)

print(f'minimal stopword removal completed')
print(f'removed {initial_len - len(df)} tweets with length < {min_length} chars')
print(f'remaining tweets: {len(df)}')
    

100%|██████████| 1882/1882 [00:00<00:00, 235014.89it/s]

minimal stopword removal completed
removed 0 tweets with length < 15 chars
remaining tweets: 1882


In [ ]:
print("="*60)
print('Preprocessing quality report')
print("="*60)

df['orig_length'] = df['full_text'].str.len()
df['final_length'] = df['final_bertopic'].str.len()

df['orig_words'] = df['full_text'].apply(lambda x: len(str(x).split()))
df['final_words'] = df['final_bertopic'].apply(lambda x: len(str(x).split()))

print(f"length statistics:")
print(f"average original length: {df['orig_length'].mean():.1f} chars")
print(f"average final length:    {df['final_length'].mean():.1f} chars")
print(f"reduction:               {((df['orig_length'].mean() - df['final_length'].mean()) / df['orig_length'].mean() * 100):.1f}%")

print(f"word count statistics:")
print(f"average original words: {df['orig_words'].mean():.1f}")
print(f"average final words:    {df['final_words'].mean():.1f}")
print(f"reduction:              {((df['orig_words'].mean() - df['final_words'].mean()) / df['orig_words'].mean() * 100):.1f}%")

print(f"distribution:")
print(f"shortest final text: {df['final_length'].min()} chars")
print(f"longest final text:  {df['final_length'].max()} chars")
print(f"median length:       {df['final_length'].median():.1f} chars")

Preprocessing quality report

 length statistics:
average original length: 167.5 chars
average final length:    139.7 chars
reduction:               16.6%

 word count statistics:
average original words: 23.9
average final words:    19.9
reduction:              16.9%

 distribution:
shortest final text: 15 chars
longest final text:  478 chars
median length:       133.0 chars


In [ ]:
bertopic_data = df[['created_at', 'user_id_str', 'full_text', 'final_bertopic']].copy()

bertopic_data['preprocessing_stats'] = bertopic_data.apply(
    lambda row: f"orig:{len(row['full_text'])}->final:{len(row['final_bertopic'])}", axis=1
)

output_file = "cleanData/bertopic_ready_x.csv"
bertopic_data.to_csv(output_file, index=False, encoding='utf-8')

print(f"Data exported to: {output_file}")
print(f"File includes: {len(bertopic_data)} tweets ready for bertopic")
print(f"Columns: {bertopic_data.columns.tolist()}")


 Data exported to: cleanData/bertopic_ready_x_csv

 File includes: 1882 tweets ready for bertopic
Columns: ['created_at', 'user_id_str', 'full_text', 'final_bertopic', 'preprocessing_stats']
